# Leksjon 3: Kryptografi og hemmeligheter

**Mål:** Forstå hvordan vi beskytter data ved hjelp av Python. Vi skal lære om **funksjoner** for å strukturere kode, **hashing** for å sikre passord, og **kryptering** for å holde hemmeligheter trygge.

## Dagens meny:
1. **Leksegjennomgang:** Ordbokangrep.
2. **Python Skills:** Funksjoner og moduler.
3. **Hashing:** Enveis-sikkerhet (MD5, SHA-256).
4. **Kryptering:** Toveis-sikkerhet (Symmetrisk kryptering).
5. **Prosjekt:** Bygge en passordbehandler.
6. **Ekte datainnbrudd:** Hvorfor "salting" er viktig.

## 1. Python Power-Up: Funksjoner

Til nå har vi skrevet script som kjører fra toppen og ned. Men hva om vi vil bruke samme kode flere ganger? Å copypaste er dårlig praksis. I stedet bruker vi **Funksjoner**.

En funksjon er en blokk med gjenbrukbar kode. Du definerer den én gang, og kaller på den når du trenger den.

### Funksjonens anatomi
```python
def funksjons_navn(parameter):
    # Kode som skal kjøres
    resultat = parameter * 2
    return resultat
```
- `def`: forteller Python at vi lager en funksjon.
- `parameter`: input-data funksjonen trenger.
- `return`: output-data funksjonen gir tilbake.

In [ ]:
# En enkel funksjon for å sjekke om passordet er langt nok
def er_passordet_langt_nok(passord):
    if len(passord) >= 8:
        return True
    else:
        return False

# Bruke funksjonen
mitt_passord = "hemmelig123"
sjekk = er_passordet_langt_nok(mitt_passord)
print(f"Er '{mitt_passord}' langt nok? {sjekk}")

kort_passord = "1234"
print(f"Er '{kort_passord}' langt nok? {er_passordet_langt_nok(kort_passord)}")

## 2. Hashing: Enveisgater

**Hashing** tar en input (som et passord) og gjør det om til en tekststreng med fast lengde (hashen).
Det viktige er: **du kan ikke gjøre hashen om til passordet igjen.** Det er en enveisprosess.

Dette er perfekt for lagring av passord. Vi lagrer ikke selve passordet; vi lagrer hashen. Når en bruker logger inn, hasher vi det de skriver inn og sammenligner med den lagrede hashen.

Vi skal bruke biblioteket `hashlib`.
*   **MD5:** Gammel, ødelagt, rask. (Ikke bruk for sikkerhet!)
*   **SHA-256:** Standard, sikker.


In [ ]:
import hashlib

passord = "superhemmeligpassord"

# MD5 Hashing (Ødelagt, kun som eksempel)
# NB: vi må kode om teksten til bytes (b"string") før vi hasher
md5_hash = hashlib.md5(passord.encode()).hexdigest()
print(f"MD5: {md5_hash}")

# SHA-256 Hashing (Sikker)
sha256_hash = hashlib.sha256(passord.encode()).hexdigest()
print(f"SHA-256: {sha256_hash}")

# Selv en bitteliten endring endrer hele hashen
feil_passord = "superhemmeligpassore" # endret 'd' til 'e'
ny_hash = hashlib.sha256(feil_passord.encode()).hexdigest()
print(f"SHA-256 (endret): {ny_hash}")

### Problemet med usaltede hasher: Rainbow Tables

Hvis to brukere har samme passord, får de samme hash. Hackere bruker "Rainbow Tables" (pre-kalkulerte lister over hasher for vanlige passord) for å knekke vanlige passord øyeblikkelig.

**Løsning: Salting.**
Et **salt** er tilfeldig data som legges til passordet før hashing. Dette gjør hashen unik selv om passordet er vanlig.


In [ ]:
import os

# Generer et tilfeldig salt (16 bytes)
salt = os.urandom(16)
print(f"Salt (bytes): {salt}")

# Kombiner salt og passord
saltet_passord = salt + passord.encode()

# Hash det saltede passordet
sikker_hash = hashlib.sha256(saltet_passord).hexdigest()
print(f"Saltet SHA-256: {sikker_hash}")

## 3. Kryptering: Toveis-sikkerhet

I motsetning til Hashing, er **Kryptering** reversibelt. Du kan låse data (kryptere) og låse det opp (dekryptere) hvis du har **Nøkkelen**.

Vi skal bruke biblioteket `cryptography`. Spesifikt **Fernet** (symmetrisk kryptering).
"Symmetrisk" betyr at *samme nøkkel* brukes til å låse og låse opp.


In [ ]:
from cryptography.fernet import Fernet

# 1. Generer en nøkkel (Som å lage en fysisk nøkkel)
key = Fernet.generate_key()
print(f"Nøkkel: {key}")

# 2. Lag låsen (Cipher suite)
cipher = Fernet(key)

# 3. Krypter en melding
melding = b"Dette er en hemmelig melding!"
kryptert_melding = cipher.encrypt(melding)
print(f"Kryptert: {kryptert_melding}")

# 4. Dekrypter meldingen
dekryptert_melding = cipher.decrypt(kryptert_melding)
print(f"Dekryptert: {dekryptert_melding}")

## 4. Prosjekt: Enkel Passordbehandler

Vi skal bygge et verktøy som:
1.  Tar et tjenestenavn (f.eks. "Netflix") og et passord.
2.  Krypterer passordet.
3.  Lagrer det i en fil.
4.  Kan lese det tilbake senere.

Vi bruker **Funksjoner** for å organisere dette.


In [ ]:
# Oppsett
key = Fernet.generate_key()
cipher = Fernet(key)
VAULT_FILE = "mine_passord.txt"

def lagre_passord(tjeneste, brukernavn, passord):
    kryptert_passord = cipher.encrypt(passord.encode()).decode() # decode bytes til string for skriving
    innslag = f"{tjeneste},{brukernavn},{kryptert_passord}\n"
    
    with open(VAULT_FILE, "a") as f:
        f.write(innslag)
    print(f"Lagret passord for {tjeneste}!")

def hent_passord(tjeneste_navn):
    # Sjekk om filen finnes først
    try:
        with open(VAULT_FILE, "r") as f:
            for linje in f:
                deler = linje.strip().split(",")
                tjeneste = deler[0]
                brukernavn = deler[1]
                tjeneste_kryptert_pass = deler[2]
                
                if tjeneste == tjeneste_navn:
                    dekryptert_pass = cipher.decrypt(tjeneste_kryptert_pass.encode()).decode()
                    print(f"Fant {tjeneste}!")
                    print(f"Bruker: {brukernavn}")
                    print(f"Passord: {dekryptert_pass}")
                    return
    except FileNotFoundError:
        print("Ingen passord lagret enda.")
        return

    print("Tjeneste ikke funnet.")

# La oss teste det!
lagre_passord("Instagram", "kul_bruker", "passord123")
lagre_passord("Bank", "rik_bruker", "sikkertPass!99")

print("\n--- Henter passord ---")
hent_passord("Instagram")

## 5. Ekte feiltrinn: Hvorfor dette betyr noe

-   **LinkedIn (2012):** lagret 6,5 millioner passord som **usaltet SHA-1 hasher**. Hackere lastet ned databasen og knekte millioner av dem raskt ved hjelp av Rainbow Tables.
-   **RockYou (2009):** Lagret 32 millioner passord i **klartekst** (ingen kryptering, ingen hashing). Alle som fikk tak i databasen så alle passordene med en gang.

**Lærdom:**
-   **Passord:** Alltid Hash + Salt.
-   **Sensitiv Data:** Krypter.
-   **Aldri** lagre passord i klartekst.


---

##  Refleksjon — Hva jeg lærte i dag

I denne leksjonen utforsket jeg de to hovedpilarene innen databeskyttelse i cybersikkerhet — hashing og kryptering — og brukte dem i et praktisk prosjekt. Jeg styrket også Python-ferdighetene mine ved å lære å skrive gjenbrukbare funksjoner.

### Viktige lærdommer

**Funksjoner og gjenbrukbarhet**
Jeg lærte å definere funksjoner med `def`, sende inn parametere og returnere verdier. Å skrive funksjoner transformerte et lineært skript til et modulært, gjenbrukbart verktøy. Passordbehandlerprosjektet demonstrerte dette tydelig: ved å kapsle inn logikken i `lagre_passord()` og `hent_passord()` ble koden renere, lettere å teste og lettere å utvide.

**Hashing: Enveis-beskyttelse**
Hashing konverterer inputdata til et fast langt sammendrag som ikke kan reverseres. Jeg forstod forskjellen mellom MD5 (rask, ødelagt, uegnet for sikkerhet) og SHA-256 (gjeldende standard). Viktigst av alt lærte jeg at hashing alene er utilstrekkelig — uten et **salt** (unike tilfeldige data lagt til før hashing) produserer identiske passord identiske hasher, noe som gjør dem sårbare for rainbow table-angrep.

**Salting**
Ved å generere et tilfeldig salt med `os.urandom()` og legge det til passordet før hashing, blir hver lagret hash unik — selv når to brukere har samme passord. Dette er den korrekte tilnærmingen for passordlagring i ethvert virkelig system.

**Symmetrisk kryptering med Fernet**
I motsetning til hashing er kryptering reversibelt. Jeg lærte å bruke `cryptography`-bibliotekets Fernet-implementasjon, som håndterer nøkkelgenerering, kryptering og dekryptering. Den samme nøkkelen må brukes til begge operasjoner — dette er definisjonen av symmetrisk kryptering.

**Bygge passordbehandleren**
Ved å kombinere funksjoner, filbehandling og kryptering bygget jeg en fungerende passordbehandler som lagrer krypterte påloggingsopplysninger og henter dem på forespørsel. Dette prosjektet knyttet sammen konseptene fra alle tre leksjonene.

**Ekte datainnbrudd**
LinkedIn- (2012) og RockYou-sakene (2009) forankret den tekniske teorien i virkelige konsekvenser. LinkedIns bruk av usaltet SHA-1 hasher lot angripere knekke millioner av passord raskt. RockYou lagret passord i klartekst — den mest kritiske feilen som er mulig. Begge eksemplene understreker hvorfor korrekte kryptografiske praksiser ikke er forhandlingsbare.

### Personlig refleksjon
Denne leksjonen var den mest teknisk krevende så langt, og også den mest givende. Jeg forstår nå den kritiske forskjellen mellom hashing (for integritet og passordlagring) og kryptering (for konfidensialitet av data som må kunne hentes igjen). Å bygge passordbehandleren gjorde disse abstrakte konseptene håndgripelige. Jeg kan nå forklare hvorfor LinkedIn-bruddet var forebyggbart, og hva den korrekte implementasjonen burde ha sett ut.